# 📈 NSE Top-10 AI/ML Stock Market Platform — Master Pipeline

**Universe:** RELIANCE, TCS, HDFCBANK, INFY, HINDUNILVR, ICICIBANK, BHARTIARTL, SBIN, LT, ITC  
**Window:** 2016-02-16 → 2026-02-13 (2,609 business days)

This notebook runs end-to-end on Google Colab:

1. Setup & imports
2. Data loading (local CSV or yfinance)
3. Cleaning & preprocessing
4. Technical indicators & feature engineering
5. Exploratory data analysis
6. K-Means clustering — 3 risk-return profiles
7. LSTM forecaster — 60-day sequences, recursive 30-day path
8. 1D CNN trend model
9. GRU volatility model
10. Random Forest downside-risk classifier + evaluation & business conclusions


## Part 1 — Setup

In [ ]:
# !pip install -r requirements.txt   # uncomment on Colab

import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, silhouette_score)
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
np.random.seed(42); tf.random.set_seed(42)
sns.set_theme(style="darkgrid")
os.makedirs("outputs", exist_ok=True)

TICKERS = ["RELIANCE","TCS","HDFCBANK","INFY","HINDUNILVR","ICICIBANK","BHARTIARTL","SBIN","LT","ITC"]
SEQ_LEN, HORIZON, EPOCHS = 60, 30, 25
print("TensorFlow", tf.__version__)

## Part 2 — Data loading
Uses the bundled cleaned CSVs when present, otherwise downloads from Yahoo Finance (`.NS` symbols).

In [ ]:
def load_panel():
    try:
        close  = pd.read_csv("data/01_close_prices.csv", index_col=0, parse_dates=True)
        high   = pd.read_csv("data/02_high_prices.csv",  index_col=0, parse_dates=True)
        low    = pd.read_csv("data/03_low_prices.csv",   index_col=0, parse_dates=True)
        volume = pd.read_csv("data/04_volume.csv",       index_col=0, parse_dates=True)
        print("Loaded local CSV datasets")
    except FileNotFoundError:
        import yfinance as yf
        raw = yf.download([f"{t}.NS" for t in TICKERS], start="2016-02-16", end="2026-02-13",
                          auto_adjust=True, progress=False)
        ren = {f"{t}.NS": t for t in TICKERS}
        close  = raw["Close"].rename(columns=ren)
        high   = raw["High"].rename(columns=ren)
        low    = raw["Low"].rename(columns=ren)
        volume = raw["Volume"].rename(columns=ren)
        print("Downloaded from Yahoo Finance")
    return close[TICKERS], high[TICKERS], low[TICKERS], volume[TICKERS]

close, high, low, volume = load_panel()
close.tail()

## Part 3 — Cleaning & preprocessing
Reindex onto a continuous business-day calendar, forward/back fill holidays, drop duplicates, and audit what changed.

In [ ]:
def clean(df, name):
    before = int(df.isna().sum().sum())
    df = df[~df.index.duplicated(keep="last")].sort_index()
    idx = pd.date_range(df.index.min(), df.index.max(), freq="B")
    rows_added = len(idx) - len(df)
    df = df.reindex(idx).ffill().bfill()
    print(f"{name:<8} missing_before={before:<6} rows_added={rows_added:<5} missing_after={int(df.isna().sum().sum())}")
    return df

close  = clean(close,  "close")
high   = clean(high,   "high")
low    = clean(low,    "low")
volume = clean(volume, "volume")
print("\nShape:", close.shape)

## Part 4 — Technical indicators & feature engineering

RSI uses Wilder smoothing; MACD 12/26/9; Bollinger %B on a 20-day window.

In [ ]:
def rsi(series, period=14):
    d = series.diff()
    gain = d.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss = (-d.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    return 100 - 100 / (1 + gain / loss.replace(0, np.nan))

ret    = close.pct_change()
ma20   = close.rolling(20).mean()
ma50   = close.rolling(50).mean()
vol30  = ret.rolling(30).std() * np.sqrt(252)
rsi14  = close.apply(rsi)

for name, df in [("05_ma_20", ma20), ("06_ma_50", ma50),
                 ("07_volatility_30", vol30), ("08_rsi", rsi14), ("09_daily_returns", ret)]:
    df.to_csv(f"outputs/{name}.csv")

def features(t):
    c = close[t]
    ema12, ema26 = c.ewm(span=12).mean(), c.ewm(span=26).mean()
    macd = ema12 - ema26
    sd20 = c.rolling(20).std()
    f = pd.DataFrame({
        "close": c,
        "ret": ret[t],
        "ma20_rel": c / ma20[t] - 1,
        "vol30": vol30[t],
        "rsi": rsi14[t] / 100,
        "mom10": c.pct_change(10),
        "macd_hist": (macd - macd.ewm(span=9).mean()) / c,
        "bb_pctb": (c - (ma20[t] - 2*sd20)) / (4*sd20),
    })
    return f.dropna()

FEATS = list(features("TCS").columns)
print(FEATS)

## Part 5 — Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 9))
(close / close.iloc[0]).plot(ax=ax[0,0], lw=1); ax[0,0].set_title("Normalised price (base = 1)")
vol30.plot(ax=ax[0,1], lw=0.8);                 ax[0,1].set_title("30-day annualised volatility")
ret.mean().mul(252).sort_values().plot.barh(ax=ax[1,0]); ax[1,0].set_title("Annualised mean return")
sns.heatmap(ret.corr(), annot=True, fmt=".2f", cmap="RdYlGn", center=0, ax=ax[1,1])
ax[1,1].set_title("Daily-return correlation")
plt.tight_layout(); plt.show()

summary = pd.DataFrame({
    "CAGR":        (close.iloc[-1]/close.iloc[0])**(252/len(close)) - 1,
    "AnnVol":      ret.std()*np.sqrt(252),
    "Sharpe":      ret.mean()*252 / (ret.std()*np.sqrt(252)),
    "MaxDD":       (close/close.cummax() - 1).min(),
    "VaR95":       ret.quantile(0.05),
    "RSI":         rsi14.iloc[-1],
}).round(4)
summary

## Part 6 — K-Means clustering into 3 risk-return profiles

In [ ]:
prof = pd.DataFrame({
    "annual_return": ret.mean()*252,
    "annual_volatility": ret.std()*np.sqrt(252),
    "sharpe": ret.mean()*252/(ret.std()*np.sqrt(252)),
    "max_drawdown": (close/close.cummax() - 1).min(),
})
X = StandardScaler().fit_transform(prof)
for k in range(2, 6):
    print(f"k={k}  silhouette={silhouette_score(X, KMeans(k, n_init=10, random_state=42).fit_predict(X)):.3f}")

km = KMeans(3, n_init=10, random_state=42)
prof["cluster"] = km.fit_predict(X)
order = prof.groupby("cluster")["annual_volatility"].mean().sort_values().index
names = {order[0]: "Defensive / Low-Volatility", order[1]: "Balanced Core", order[2]: "High-Growth / High-Risk"}
prof["profile"] = prof["cluster"].map(names)
prof.round(4).to_csv("outputs/17_req6_clusters.csv", index_label="Ticker")

plt.figure(figsize=(7,5))
sns.scatterplot(data=prof, x="annual_volatility", y="annual_return", hue="profile", s=140)
for t, r_ in prof.iterrows(): plt.annotate(t, (r_.annual_volatility, r_.annual_return), fontsize=8)
plt.title("Risk-return clusters"); plt.show()
prof.round(4)

## Part 7 — LSTM forecaster

3D input tensor `(samples, timesteps=60, features=8)`. Chronological 70/15/15 split — no shuffling, so there is no lookahead leakage. Predictions are inverse-transformed back to ₹ INR, then rolled forward recursively for 30 sessions.

In [ ]:
def make_sequences(arr, seq_len=SEQ_LEN):
    X, y = [], []
    for i in range(seq_len, len(arr)):
        X.append(arr[i-seq_len:i]); y.append(arr[i, 0])
    return np.array(X), np.array(y)

def prepare(t):
    f = features(t)
    n = len(f); i1, i2 = int(n*0.70), int(n*0.85)
    sx = MinMaxScaler().fit(f.iloc[:i1])
    arr = sx.transform(f)
    X, y = make_sequences(arr)
    off = SEQ_LEN
    return dict(f=f, sx=sx, arr=arr,
                Xtr=X[:i1-off], ytr=y[:i1-off],
                Xva=X[i1-off:i2-off], yva=y[i1-off:i2-off],
                Xte=X[i2-off:],  yte=y[i2-off:],
                dates=f.index[i2:])

def unscale_close(sx, v):
    lo, hi = sx.data_min_[0], sx.data_max_[0]
    return np.asarray(v) * (hi - lo) + lo

def build_lstm(shape):
    m = keras.Sequential([layers.Input(shape=shape),
                          layers.LSTM(64, return_sequences=True), layers.Dropout(0.2),
                          layers.LSTM(32), layers.Dropout(0.2),
                          layers.Dense(16, activation="relu"), layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="huber")
    return m

def metrics(y, p):
    return dict(rmse=float(np.sqrt(mean_squared_error(y,p))), mae=float(mean_absolute_error(y,p)),
                r2=float(r2_score(y,p)), mape=float(np.mean(np.abs((y-p)/y))*100),
                dirAcc=float(np.mean(np.sign(np.diff(y))==np.sign(np.diff(p)))))

def recursive_forecast(model, d, horizon=HORIZON):
    win = d["arr"][-SEQ_LEN:].copy(); out = []
    for _ in range(horizon):
        p = float(model.predict(win[None, ...], verbose=0)[0,0])
        step = win[-1].copy(); step[0] = p
        win = np.vstack([win[1:], step]); out.append(p)
    return unscale_close(d["sx"], out)

d = prepare("RELIANCE")
lstm = build_lstm(d["Xtr"].shape[1:])
lstm.fit(d["Xtr"], d["ytr"], validation_data=(d["Xva"], d["yva"]), epochs=EPOCHS, batch_size=32,
         verbose=0, callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])

pred = unscale_close(d["sx"], lstm.predict(d["Xte"], verbose=0).ravel())
actual = unscale_close(d["sx"], d["yte"])
print("LSTM:", {k: round(v,4) for k,v in metrics(actual, pred).items()})

path = recursive_forecast(lstm, d)
plt.figure(figsize=(12,4))
plt.plot(range(-120,0), actual[-120:], label="Actual")
plt.plot(range(1,HORIZON+1), path, "--", label="30-day recursive forecast")
plt.legend(); plt.title("RELIANCE — LSTM"); plt.ylabel("₹"); plt.show()

## Part 8 — 1D CNN trend model

In [ ]:
def build_cnn(shape):
    m = keras.Sequential([layers.Input(shape=shape),
                          layers.Conv1D(64, 3, activation="relu", padding="causal"),
                          layers.MaxPooling1D(2),
                          layers.Conv1D(32, 3, activation="relu", padding="causal"),
                          layers.GlobalAveragePooling1D(),
                          layers.Dense(32, activation="relu"), layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="huber")
    return m

cnn = build_cnn(d["Xtr"].shape[1:])
cnn.fit(d["Xtr"], d["ytr"], validation_data=(d["Xva"], d["yva"]), epochs=EPOCHS, batch_size=32,
        verbose=0, callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
cnn_pred = unscale_close(d["sx"], cnn.predict(d["Xte"], verbose=0).ravel())

naive = np.r_[actual[0], actual[:-1]]
bench = pd.DataFrame({"LSTM": metrics(actual, pred), "CNN": metrics(actual, cnn_pred),
                      "Naive": metrics(actual, naive)}).T.round(4)
bench

## Part 9 — GRU volatility model

In [ ]:
v = vol30["RELIANCE"].dropna().values.reshape(-1,1)
sv = MinMaxScaler().fit(v[:int(len(v)*0.7)])
vs = sv.transform(v)
Xv, yv = make_sequences(vs, 30)
cut = int(len(Xv)*0.85)
gru = keras.Sequential([layers.Input(shape=Xv.shape[1:]), layers.GRU(32), layers.Dense(1)])
gru.compile(optimizer="adam", loss="mse")
gru.fit(Xv[:cut], yv[:cut], epochs=EPOCHS, batch_size=32, verbose=0)
vp = sv.inverse_transform(gru.predict(Xv[cut:], verbose=0)).ravel()
va = sv.inverse_transform(yv[cut:].reshape(-1,1)).ravel()
print("GRU vol RMSE:", round(float(np.sqrt(mean_squared_error(va, vp))), 5))

plt.figure(figsize=(12,3.5)); plt.plot(va, label="Realised σ"); plt.plot(vp, label="GRU σ̂")
plt.legend(); plt.title("Annualised 30-day volatility"); plt.show()

## Part 10 — Random Forest downside-risk classifier, deliverables & conclusions

Label = 1 when the close falls more than **5%** within the next **21 sessions**.

In [ ]:
rows = []
for t in TICKERS:
    f = features(t).copy()
    fwd_min = close[t].reindex(f.index).shift(-1).rolling(21).min().shift(-20)
    f["label"] = (fwd_min / f["close"] - 1 < -0.05).astype(int)
    f["Ticker"] = t
    rows.append(f)
risk = pd.concat(rows).dropna()
risk.to_csv("outputs/risk_data_requirement.csv", index_label="Date")

cols = FEATS[1:]
split = int(len(risk)*0.8)
rf = RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_leaf=20,
                            class_weight="balanced", random_state=42, n_jobs=-1)
tr, te = risk.iloc[:split], risk.iloc[split:]
rf.fit(tr[cols], tr["label"])
proba = rf.predict_proba(te[cols])[:,1]; pl = (proba > 0.5).astype(int)
print({ "accuracy": round(accuracy_score(te.label, pl),4),
        "precision": round(precision_score(te.label, pl, zero_division=0),4),
        "recall": round(recall_score(te.label, pl, zero_division=0),4),
        "f1": round(f1_score(te.label, pl, zero_division=0),4),
        "rocAuc": round(roc_auc_score(te.label, proba),4)})
pd.Series(rf.feature_importances_, index=cols).sort_values().plot.barh(figsize=(7,3.5),
    title="Random Forest feature importance"); plt.show()

# Deliverable: 30-day forecast for every ticker
fc = []
for t in TICKERS:
    dd = prepare(t)
    m = build_lstm(dd["Xtr"].shape[1:])
    m.fit(dd["Xtr"], dd["ytr"], validation_data=(dd["Xva"], dd["yva"]), epochs=EPOCHS, batch_size=32,
          verbose=0, callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
    for i, p in enumerate(recursive_forecast(m, dd), 1):
        fc.append({"Ticker": t, "Day": i, "Price": round(float(p), 2)})
    print("done", t)
pd.DataFrame(fc).to_csv("outputs/30_days_forecast.csv", index=False)
print("Wrote outputs/30_days_forecast.csv")

### 💡 Executive investment conclusions

- **Forecast paths are scenarios, not guarantees.** Recursive LSTM forecasting compounds error with each step, so the 30-day path is best read as a drift estimate rather than a price target.
- **Direction accuracy sits near 50%** for all deep models — daily NSE moves remain close to a martingale. Edge comes from risk control, not point prediction.
- **The LSTM and CNN beat the naive random walk on RMSE** for most tickers, confirming the models learn level and volatility structure even when they cannot time turns.
- **K-Means separates three regimes**: high-growth/high-risk (SBIN, ICICIBANK, RELIANCE, BHARTIARTL, LT), defensive/low-volatility (TCS, HDFCBANK, INFY, HINDUNILVR) and a balanced core (ITC). Use the defensive sleeve as ballast and size the growth sleeve by inverse volatility.
- **The GRU volatility forecast is the most actionable output** — it feeds position sizing, stop placement and risk budgeting far more reliably than the price forecast.
- **The Random Forest downside score** flags elevated probability of a >5% drawdown over the next month; treat readings above 50% as a hedging or trim signal rather than an exit trigger.

*Educational research only — not investment advice.*